# Extensión A: más modelos de oraciones y sus protocolos

**Materiales desarrollados por Matías Barreto, 2026**

**Tecnicatura Superior en Ciencias de Datos e IA, IFTS N.º 24**
* **Nomenclatura oficial:** Procesamiento de Lenguaje Natural
* **Nombre de trabajo:** Laboratorio de PLN: Analítica, Textos y Cultura

---

## Pregunta

¿Cambian las conclusiones del cuaderno 05 si usamos modelos más grandes, y qué pasa si ignoramos su protocolo?

## Objetivo

Repetir la evaluación de recuperación con otros dos modelos multilingües, uno por vez, y medir el efecto de omitir los prefijos que uno de ellos exige.

## Al terminar este cuaderno vamos a poder:

1. Repetir una evaluación con modelos distintos sin cambiar la colección, las consultas ni los juicios.
2. Leer la ficha de un modelo antes de usarlo y comprobar el efecto de ignorar su protocolo.
3. Registrar tamaño, licencia y costo de cómputo junto con las métricas.

## Prerrequisitos reales

* Cuadernos 04 y 05 de esta unidad (embeddings de oraciones y evaluación de recuperación).

## Recursos y condiciones de ejecución

Local. **Descarga unos 4,5 GB** la primera vez (dos modelos de ~2 GB cada uno) y tarda varios minutos por modelo en CPU. Se carga **un modelo por vez**: cada uno se descarta antes de cargar el siguiente. Necesita unos 4 GB libres de memoria.

## Qué es esencial y qué es opcional

* **Esencial:** Todo el cuaderno es opcional: es una extensión del cuaderno 05.
* **Opcional:** Todo.

## Producto

Una tabla con las métricas de los cuatro modos y tres líneas sobre qué cambió y qué no puede afirmarse.

## Alcance

Cuatro configuraciones sobre 10 consultas y 137 fragmentos. No permite decir qué modelo es mejor en general.

In [1]:
# Esta extensión vive en extensiones/. Subimos a la carpeta de la unidad para que las rutas relativas funcionen.
import os
from pathlib import Path

if Path.cwd().name == "extensiones":
    os.chdir(Path.cwd().parent)
print("Carpeta de trabajo:", Path.cwd().name)

Carpeta de trabajo: 009_representaciones_semanticas


## 1. Los modelos y su protocolo

Verificamos cada ficha en Hugging Face (consulta del 7 de octubre de 2026).

| | `paraphrase-multilingual-MiniLM-L12-v2` | `intfloat/multilingual-e5-large` | `BAAI/bge-m3` |
|---|---|---|---|
| Licencia | Apache 2.0 | MIT | MIT |
| Dimensiones del vector | 384 | 1024 | 1024 |
| Longitud máxima | 128 tokens | 512 tokens | 8192 tokens |
| Prefijos | No requiere | **Obligatorios**: `query: ` y `passage: ` | No requiere instrucciones |
| Tamaño aproximado | ~470 MB | ~2,2 GB | ~2,3 GB |
| Parámetros | 117,7 millones | 559,9 millones | no verificado |

La ficha de E5 dice, textualmente, que cada texto debe empezar con "query: " o "passage: ", y que de lo contrario se observa una degradación del rendimiento. Para recuperación asimétrica (una consulta corta contra textos largos), las consultas llevan `query: ` y los fragmentos llevan `passage: `.

## 2. Configuración

In [2]:
import gc
import sys
import time
import importlib.metadata

import numpy as np
import pandas as pd
from IPython.display import display

from sentence_transformers import SentenceTransformer

import utils

pd.set_option("display.max_colwidth", 90)

print("Python:", sys.version.split()[0])
for paquete in ["sentence-transformers", "numpy", "pandas"]:
    print(f"{paquete}: {importlib.metadata.version(paquete)}")

fragmentos = utils.cargar_fragmentos()
consultas = utils.cargar_consultas()
juicios = utils.cargar_juicios()
lista_de_ids = list(fragmentos["id"])
textos_de_la_colección = list(fragmentos["texto"])
print(len(fragmentos), "fragmentos |", len(consultas), "consultas |", len(juicios), "juicios")

Python: 3.12.13
sentence-transformers: 6.1.0
numpy: 2.5.3
pandas: 3.0.6
137 fragmentos | 10 consultas | 83 juicios


## 3. Una función para cada modo

Cada modo es: un modelo, un prefijo para las consultas y un prefijo para los fragmentos. Evaluamos con las mismas métricas del cuaderno 05 (precisión@3 y rango recíproco, aceptando parciales o solo totales).

In [3]:
def evaluar_un_modo(nombre_del_modelo, prefijo_de_consulta, prefijo_de_fragmento):
    """Carga un modelo, evalúa la recuperación y lo descarta. Devuelve las métricas promedio y el tiempo."""
    inicio = time.perf_counter()
    modelo = SentenceTransformer(nombre_del_modelo)

    textos_con_prefijo = []
    for texto in textos_de_la_colección:
        textos_con_prefijo.append(prefijo_de_fragmento + texto)
    matriz_de_la_colección = modelo.encode(textos_con_prefijo, batch_size=8, show_progress_bar=False)

    filas = []
    for id_consulta in consultas["id"]:
        texto_de_la_consulta = consultas.loc[consultas["id"] == id_consulta, "consulta"].iloc[0]
        vector_de_la_consulta = modelo.encode([prefijo_de_consulta + texto_de_la_consulta], show_progress_bar=False)[0]
        puntajes = utils.cosenos_contra_todas_las_filas(matriz_de_la_colección, vector_de_la_consulta)
        relevancias = utils.relevancias_de_una_consulta(lista_de_ids, juicios, id_consulta)
        relevancias_ordenadas = relevancias[utils.ordenar_por_puntaje(puntajes)]
        filas.append({
            "P@3 (≥1)": utils.precision_en_k(relevancias_ordenadas, k=3, umbral=1),
            "P@3 (=2)": utils.precision_en_k(relevancias_ordenadas, k=3, umbral=2),
            "RR (≥1)": utils.rango_reciproco(relevancias_ordenadas, umbral=1),
            "RR (=2)": utils.rango_reciproco(relevancias_ordenadas, umbral=2),
        })
    promedios = pd.DataFrame(filas).mean().round(2).to_dict()
    promedios["segundos"] = round(time.perf_counter() - inicio)
    promedios["longitud máxima"] = modelo.max_seq_length

    del modelo
    gc.collect()
    return promedios

### ✎ Antes de ejecutar

Hay cuatro modos. ¿Qué esperan de cada uno respecto de MiniLM? ¿Y del modo E5 **sin** prefijos, comparado con E5 **con** prefijos? Anoten una predicción antes de correr la celda siguiente (tarda varios minutos).

> _(Escriban acá.)_

In [4]:
modos = [
    ("MiniLM (128 tokens)", "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", "", ""),
    ("E5 sin prefijos (ignora el protocolo)", "intfloat/multilingual-e5-large", "", ""),
    ("E5 con prefijos (según la ficha)", "intfloat/multilingual-e5-large", "query: ", "passage: "),
    ("BGE-M3", "BAAI/bge-m3", "", ""),
]

resultados = {}
for nombre_del_modo, nombre_del_modelo, prefijo_c, prefijo_f in modos:
    print("Evaluando:", nombre_del_modo)
    resultados[nombre_del_modo] = evaluar_un_modo(nombre_del_modelo, prefijo_c, prefijo_f)

display(pd.DataFrame(resultados).T)

Evaluando: MiniLM (128 tokens)


Evaluando: E5 sin prefijos (ignora el protocolo)


Evaluando: E5 con prefijos (según la ficha)


Evaluando: BGE-M3


,P@3 (≥1),P@3 (=2),RR (≥1),RR (=2),segundos,longitud máxima
MiniLM (128 tokens),0.43,0.30,0.74,0.58,5.0,128.0
E5 sin prefijos (ignora el protocolo),0.50,0.40,0.77,0.75,10.0,512.0
E5 con prefijos (según la ficha),0.60,0.43,0.72,0.72,9.0,512.0
BGE-M3,0.60,0.43,0.75,0.72,10.0,8192.0


**Lectura.**

* **Observación.** Con P@3 (≥1), los dos modelos grandes con su protocolo (E5 con prefijos y BGE-M3) llegan a 0,60 frente a 0,43 de MiniLM. Con el rango recíproco las diferencias son mucho menores: entre 0,72 y 0,77 en los tres modos grandes y 0,74 en MiniLM. E5 sin prefijos da 0,50 en P@3 (≥1), por debajo de E5 con prefijos, y 0,75 en RR (=2), apenas por encima.
* **Interpretación posible.** Los modelos grandes ponen más fragmentos relevantes entre los tres primeros, pero no necesariamente ponen el primero antes. Omitir los prefijos coincide con una baja en P@3 (≥1), en el sentido que advierte la ficha de E5; con 10 consultas, no alcanza para decir cuánto. Parte de la diferencia con MiniLM puede deberse a que los modelos grandes leen hasta 512 tokens y MiniLM solo 128 (10 fragmentos se truncan), pero este cuaderno no lo separa de los demás factores.
* **Límite.** La columna `segundos` de esta ejecución no incluye descargas porque los modelos ya estaban en caché. En la primera ejecución, con descarga, E5 tardó 73 s y BGE-M3 80 s. Las métricas fueron idénticas en las dos ejecuciones.

## 4. Cierre

**Qué podemos afirmar y qué no.**

* Cada modo se evaluó **una vez**, con las mismas 10 consultas y los mismos juicios. Una diferencia de 0,05 a 0,10 entre modos cae dentro del ruido que medimos en el cuaderno 05.
* Omitir los prefijos de E5 es **cambiar el modo de uso del modelo**: la comparación "E5 sin prefijos" mide un uso incorrecto, no el modelo.
* Más parámetros, más dimensiones o más tokens no garantizan mejores resultados en una colección y unas consultas concretas.
* Tamaño y costo importan: cada modo tarda segundos distintos y ocupa memoria distinta. Una decisión real los pesa junto con la calidad.

> _(Escriban acá su conclusión, con un caso favorable, un contraejemplo y una decisión.)_